In [1]:
import numpy as np
from astropy.time import Time
from astropy.coordinates import EarthLocation, get_body_barycentric_posvel, solar_system_ephemeris 
import astropy.units as u

In [2]:
%%timeit

a = []
for x in range(0, 100000):
    a.append(x)

1.14 ms ± 107 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [3]:
%%timeit

a = np.arange(100000)

23.1 μs ± 4.65 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [4]:
%%timeit

arrA = [ x for x in range(0, 100000) ]
arrB = [ x for x in range(0, 100000) ]

result = []
for a, b in zip(arrA, arrB):
    result.append(a + b)

3.84 ms ± 593 ns per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [5]:
%%timeit

arrA = np.arange(0, 100000)
arrB = np.arange(0, 100000)

result = arrA + arrB

68.5 μs ± 239 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [6]:
from numpy import sin, cos
import numpy as np

def Sky_basis(ra_rad, dec_rad):
        l_hat = np.array([cos(dec_rad)*cos(ra_rad), # vector that points to object's pos on sky from observer
                          cos(dec_rad)*sin(ra_rad),
                          sin(dec_rad)])
        e_ra  = np.array([-sin(ra_rad),  cos(ra_rad), 0.0]) # vector towards increasing RA - points east, 90 deg to l_hat
        e_dec = np.array([-sin(dec_rad)*cos(ra_rad), # vector towards increasing dec - points north
                          -sin(dec_rad)*sin(ra_rad),
                          cos(dec_rad)])
        return l_hat, e_ra, e_dec # turning angular rates into linear velocities

AU_km = 149_597_870.7

def compute_unit_vectors(ra_deg, dec_deg, dra_deg_per_day, ddec_deg_per_day):
        mu_sun = 1.32712440018e11  # [km^3/s^2] gm of the sun

        # converting from degrees to radians
        ra  = np.deg2rad(ra_deg) 
        dec = np.deg2rad(dec_deg)
        l_hat, e_ra, e_dec = Sky_basis(ra, dec)
         # angular rate conversion from deg/day to rad/s
        dra  = np.deg2rad(dra_deg_per_day)  / 86400.0
        ddec = np.deg2rad(ddec_deg_per_day) / 86400.0

        v_hat = (dra * e_ra + ddec * e_dec)

        return l_hat, v_hat

def observables_to_geocentric_state(l_hat, v_hat, d_au, ddot_kms):   
        d_km = d_au * AU_km

        # multiplying l_hat by the rand dist. value gives actual geocentric pos vector of obj
        r_geo = np.outer(d_km, l_hat)

        # speed
        v_rad = np.outer(ddot_kms, l_hat)
        v_tan = np.outer(d_km, v_hat)
        v_geo = v_rad + v_tan
        
        return r_geo, v_geo

In [7]:
%%timeit
observables_to_geocentric_state(10, 20, 0.12, 0.22, 1., 4)

TypeError: observables_to_geocentric_state() takes 4 positional arguments but 6 were given

In [ ]:
l_hat, v_hat = compute_unit_vectors(10, 20, 0.12, 0.22)

In [ ]:
%%timeit
observables_to_geocentric_state(l_hat, v_hat, 1., 4)

In [ ]:
d_au = np.arange(0, 2, 0.1)
ddot_kms = np.arange(3, 5, 0.1)

In [ ]:
d_grid, ddot_grid = np.meshgrid(d_au, ddot_kms)
d_grid, ddot_grid = d_grid.flatten(), ddot_grid.flatten()

In [ ]:
%%timeit
observables_to_geocentric_state(l_hat, v_hat, d_grid, ddot_grid)

In [ ]:
len(d_grid)

In [ ]:
r_geo, v_geo = observables_to_geocentric_state(l_hat, v_hat, d_grid, ddot_grid)

In [ ]:
r_geo.shape

In [ ]:
d_grid[10], ddot_grid[10], r_geo[10], v_geo[10]

In [ ]:
def get_earth_and_observer(obstime_str):
    obstime = Time(obstime_str)
    
    rubin_location = EarthLocation.from_geodetic( 
        lon=-70.7366*u.deg,   # longitude (west is negative)
        lat=-30.2407*u.deg,   # latitude
        height=2647*u.m       # elevation
        
    )
    obs_geo = rubin_location.get_gcrs(obstime)
    r_obs = np.array([obs_geo.cartesian.x.to(u.km).value,
                      obs_geo.cartesian.y.to(u.km).value,
                      obs_geo.cartesian.z.to(u.km).value])
    
    # earth's barycentric vector rel to sun aka center of solsys to earth
    with solar_system_ephemeris.set('de432s'): 
        rE_bary, vE_bary = get_body_barycentric_posvel('earth', obstime)

    # convert to km and km/s arrays in cartesian coordinate system
    rE = np.array([rE_bary.x.to(u.km).value, #rE goes from sun to earth center
                   rE_bary.y.to(u.km).value,
                   rE_bary.z.to(u.km).value])
    
    vE = np.array([vE_bary.x.to(u.km/u.s).value, #vE goes from sun to earth center
                   vE_bary.y.to(u.km/u.s).value,
                   vE_bary.z.to(u.km/u.s).value])
    return obstime, rE, vE, r_obs

In [ ]:
%%timeit
get_earth_and_observer("2025-09-16T00:00:00")

In [ ]:
rubin_location = EarthLocation.from_geodetic( 
        lon=-70.7366*u.deg,   # longitude (west is negative)
        lat=-30.2407*u.deg,   # latitude
        height=2647*u.m       # elevation
        
    )

        

def get_earth_and_observer(obstime_str):
    obstime = Time(obstime_str)
    
    obs_geo = rubin_location.get_gcrs(obstime)
    r_obs =  obs_geo.cartesian.xyz.to_value(u.km).T
    
    # earth's barycentric vector rel to sun aka center of solsys to earth
    with solar_system_ephemeris.set('de432s'): 
        rE_bary, vE_bary = get_body_barycentric_posvel('earth', obstime)

    # convert to km and km/s arrays in cartesian coordinate system
    rE  = rE_bary.xyz.to_value(u.km).T #rE goes from sun to earth center
    vE  = vE_bary.xyz.to_value(u.km/u.s).T

      #vE goes from sun to earth center

    return obstime, rE, vE, r_obs

In [ ]:
%%timeit
get_earth_and_observer("2025-09-16T00:00:00")

In [ ]:
obstime, rE, vE, r_obs = get_earth_and_observer("2025-09-16T00:00:00")


In [ ]:
rE

In [ ]:
def sky_basis(ra_rad, dec_rad):
        l_hat = np.array([np.cos(dec_rad)*np.cos(ra_rad), # vector that points to object's pos on sky from observer
                          np.cos(dec_rad)*np.sin(ra_rad),
                          np.sin(dec_rad)])
        e_ra  = np.array([-np.sin(ra_rad), np.cos(ra_rad), 0.0]) # vector towards increasing RA - points east, 90 deg to l_hat
        e_dec = np.array([-np.sin(dec_rad)*np.cos(ra_rad), # vector towards increasing dec - points north
                          -np.sin(dec_rad)*np.sin(ra_rad),
                          np.cos(dec_rad)])
        return l_hat, e_ra, e_dec # turning angular rates into linear velocities


In [ ]:
%%timeit
sky_basis(ra_rad = 0.1745, dec_rad = 0.3491)

In [ ]:
def geo_to_topo(r_geo_km, v_geo_kms, rE, vE, r_obs, obstime):
        # convert r_geo, v_geo for obj to heliocentric r_topo, v_topo for an observer at rubin
        # from center of earth to an observer now at rubin
        obstime = Time(obstime)
        # earth's barycentric (heliocentric) state was done in previous step
        # so was observer's offset from center of earth
        # adding the offsets
        r_topo = rE + r_obs + r_geo_km
        v_topo = vE + v_geo_kms
        return r_topo, v_topo # asteroid position, vec going from sun to asteroid assuming youre at rubin

In [ ]:
%%timeit
geo_to_topo(r_geo, v_geo, rE, vE, r_obs, obstime)